# Official OMP (mergekit-tokensurgeon) against calibrated ridge, on the same tokenizer-transplant task
**Task (the one the OMP paper uses):** take a base model (GPT-2, GPT-2 medium or Qwen2.5-0.5B), swap in the tokenizer of a donor model (Pythia, Qwen or GPT-2) and rebuild the embedding table. Tokens that exist in both vocabularies keep the base embedding; every other token has to be approximated.
The official tool `mergekit-tokensurgeon` does the approximation (default: OMP with k=64). We also run its other official methods, and fill the SAME missing tokens with our calibrated ridge, so the only difference between methods is how the missing rows are built.
Evaluation: total NLL of the same sentences (PubMed text = domain, WikiText = general). Total sentence NLL is comparable across tokenizers. Zero-shot, no training. Lower is better. Reference: the unchanged base model with its own tokenizer.
Turn Internet and GPU on. Run top to bottom. Time: about 1 hour per pair (estimate, untested on Kaggle); each method is saved as soon as it finishes, so a rerun resumes.

In [ ]:
%pip install -q mergekit deepfocus datasets scikit-learn

In [ ]:
import re, time, gc, random, pickle, subprocess, shutil, traceback, os, json
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be very slow)")

## Config

In [ ]:
N_ABSTRACTS = 4000; TRAIN_FRAC = 0.7
N_CAL = 150                 # calibration sentences per text type (domain, general), from the training part
N_EVAL = 400                # evaluation sentences per text type
GRID = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5]
OMP_KS = (64, 32)           # 64 is the tool's default and the paper's setting
EXTRA_OFFICIAL = [("subword-mean", ["-a", "subword", "-s", "mean"]), ("common-interp", ["-a", "common_interpolation"]),
                  ("john-hewitt", ["-a", "john_hewitt"]), ("mean", ["-a", "mean"])]
EVAL_BATCH = 8
FOCUS_MIN_COUNT = 10        # fastText minimum token count inside FOCUS (the package default)
FOCUS_TEXT_MAX = 60000      # sentences used to train FOCUS's auxiliary fastText model (training text only)
DOMAIN_TXT = None; USE_WIKI = True
OUT = Path("official_results"); OUT.mkdir(exist_ok=True)
WORK = Path("ts_work"); WORK.mkdir(exist_ok=True)
PAIRS = [   # tied-embedding bases that mergekit-tokensurgeon supports (it has no OPT or GPT-Neo architecture file). The base keeps its weights and gets the donor's tokenizer.
    dict(name="GPT-2 <- Pythia tokenizer",         base="openai-community/gpt2",        donor="EleutherAI/pythia-160m"),
    dict(name="GPT-2 <- Qwen tokenizer",           base="openai-community/gpt2",        donor="Qwen/Qwen2.5-0.5B"),
    dict(name="GPT-2 medium <- Pythia tokenizer",  base="openai-community/gpt2-medium", donor="EleutherAI/pythia-160m"),
    dict(name="Qwen2.5-0.5B <- GPT-2 tokenizer",   base="Qwen/Qwen2.5-0.5B",            donor="openai-community/gpt2"),
]

## Text

In [ ]:
def split_sents(text):
    text = " ".join(text.split())
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if 8 <= len(s.split()) <= 60]

def get_domain():
    out = []
    if DOMAIN_TXT: out = [l.strip() for l in open(DOMAIN_TXT, encoding="utf-8") if l.strip()][:N_ABSTRACTS]
    else:
        from datasets import load_dataset
        for cfg, stream in (("pqa_artificial", True), ("pqa_labeled", False)):
            try:
                for ex in load_dataset("qiaojin/PubMedQA", cfg, split="train", streaming=stream):
                    out.append(" ".join(ex["context"]["contexts"]))
                    if len(out) >= N_ABSTRACTS: break
                if out: print(f"domain text: PubMedQA/{cfg}, {len(out)} abstracts"); break
            except Exception as e: print(f"PubMedQA/{cfg} failed: {type(e).__name__}: {str(e)[:150]}")
    if not out: raise RuntimeError("Could not load the domain text. Turn Internet on, or set DOMAIN_TXT.")
    return out

def get_wiki():
    if not USE_WIKI: return []
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        s = split_sents(" ".join(x for x in ds["text"][:30000] if x.strip() and not x.startswith(" =")))
        print(f"general text: WikiText-2, {len(s)} sentences"); return s
    except Exception as e: print(f"WikiText-2 not available ({type(e).__name__})"); return []

def make_text():
    ab = get_domain(); n = int(len(ab) * TRAIN_FRAC)
    tr = [s for a in ab[:n] for s in split_sents(a)]; ev = [s for a in ab[n:] for s in split_sents(a)]
    wk = get_wiki(); rng = random.Random(0)
    smp = lambda L, k: rng.sample(L, min(k, len(L)))
    T = dict(cal_dom=smp(tr, N_CAL), eval_dom=smp(ev, N_EVAL), cal_gen=smp(wk[:len(wk) // 2], N_CAL) if wk else [], eval_gen=smp(wk[len(wk) // 2:], N_EVAL) if wk else [])
    T["train_text"] = tr[:3000] + (wk[:len(wk) // 2][:3000] if wk else [])
    T["fasttext_text"] = (tr + (wk[:len(wk) // 2] if wk else []))[:FOCUS_TEXT_MAX]
    print({k: len(v) for k, v in T.items()}); return T
TEXT = make_text()
FOCUS_JSONL = WORK / "focus_train.jsonl"
with open(FOCUS_JSONL, "w", encoding="utf-8") as fh:
    for t_ in TEXT["fasttext_text"]: fh.write(json.dumps({"text": t_}) + "\n")

## Scoring and the ridge fill

In [ ]:
def load_lm(path):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForCausalLM.from_pretrained(path).float().to(DEVICE).eval()
    return tok, m

@torch.no_grad()
def sent_nll(model, tok, sents):
    """Total NLL of each sentence under the model with its own tokenizer (the first token is predicted from a BOS token)."""
    V = len(tok); bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    ids = [tok.encode(s, add_special_tokens=False) for s in sents]; order = sorted(range(len(ids)), key=lambda i: len(ids[i])); out = np.zeros(len(ids))
    for b in range(0, len(order), EVAL_BATCH):
        idx = order[b:b + EVAL_BATCH]; L = max(len(ids[i]) for i in idx) + 1
        x = torch.full((len(idx), L), bos, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx): x[r, 1:1 + len(ids[i])] = torch.tensor(ids[i]); m[r, :1 + len(ids[i])] = 1
        x, m = x.to(DEVICE), m.to(DEVICE)
        lg = model(input_ids=x, attention_mask=m).logits[:, :-1, :V].float()
        nl = F.cross_entropy(lg.reshape(-1, V), x[:, 1:].reshape(-1), reduction="none").view(len(idx), -1)
        out[idx] = (nl * m[:, 1:].float()).sum(1).cpu().numpy()
    return out

def score_all(model, tok):
    return {k: sent_nll(model, tok, TEXT[k]) for k in ("eval_dom", "eval_gen") if TEXT[k]}

def unit(x, eps=1e-12): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def fit_ridge(Dc, Rc, seed=0):
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in (1e-4, 1e-3, 1e-2, 1e-1, 1.0):
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return solve(X, Y, best[1]), s, best[1]

def ridge_dirs(Dc, Rc, Dz_c):
    W, s, lam = fit_ridge(Dc, Rc); return unit((Dz_c / s) @ W), (Dz_c / s) @ W, lam

def set_rows(model, Z, rows):
    w = model.get_input_embeddings().weight
    with torch.no_grad(): w[torch.as_tensor(Z, device=w.device)] = torch.as_tensor(np.asarray(rows, np.float32), device=w.device)

def cal_objective(model, tok): return float(np.mean([sent_nll(model, tok, TEXT[k]).mean() for k in ("cal_dom", "cal_gen") if TEXT[k]]))

def calibrate(model, tok, Z, mu_r, rbar, dirs):
    best = None
    for c in GRID:
        set_rows(model, Z, mu_r + c * rbar * dirs); o = cal_objective(model, tok)
        if best is None or o < best[1]: best = (c, o)
    return best

def fp32_path(mid):
    """Local float32 copy of a model: some checkpoints (Qwen) are stored in bfloat16, and mergekit-tokensurgeon then fails with a dtype mismatch."""
    p = WORK / "fp32" / str(mid).strip("/").replace("/", "__")
    if not (p / "config.json").exists():
        tok = AutoTokenizer.from_pretrained(mid); m = AutoModelForCausalLM.from_pretrained(mid).float()
        p.mkdir(parents=True, exist_ok=True); m.save_pretrained(p); tok.save_pretrained(p); del m; gc.collect()
    return str(p)

def run_ts(base, donor, outdir, args):
    if Path(outdir).exists(): shutil.rmtree(outdir)
    cmd = ["mergekit-tokensurgeon", base, donor, str(outdir), "--allow-crimes"] + (["--cuda"] if DEVICE == "cuda" else []) + args
    t = time.time(); p = subprocess.run(cmd, capture_output=True, text=True); dt = time.time() - t
    if p.returncode != 0: raise RuntimeError("mergekit-tokensurgeon failed:\n" + (p.stdout + p.stderr)[-1500:])
    return dt

## Run (each method is saved when done; official FOCUS is included)

In [ ]:
def run_pair(pair):
    pair = dict(pair, base=fp32_path(pair["base"]), donor=fp32_path(pair["donor"]))
    name = pair["name"]; f = OUT / (name.replace(" ", "_").replace("<-", "from") + ".pkl")
    R = pickle.load(open(f, "rb")) if f.exists() else dict(pair=name, scores={}, secs={}, info={})
    save = lambda: pickle.dump(R, open(f, "wb"))
    OMPF = WORK / ("omp_emb_" + re.sub(r"\W+", "_", name) + ".npy")      # the official OMP rows of THIS pair
    print("\n" + "#" * 20, name, "#" * 20)
    # reference: the unchanged base with its own tokenizer
    if "base (own tokenizer)" not in R["scores"]:
        tok, m = load_lm(pair["base"]); R["scores"]["base (own tokenizer)"] = score_all(m, tok); del m; gc.collect(); torch.cuda.empty_cache(); save()
    print(f"  base reference: " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"]["base (own tokenizer)"].items()))
    # official methods
    methods = [(f"OMP-{k} (official)", ["-a", "omp", "-k", str(k)]) for k in OMP_KS] + [(f"{n} (official)", a) for n, a in EXTRA_OFFICIAL]
    for mname, args in methods:
        if mname in R["scores"]: continue
        try:
            R["secs"][mname] = run_ts(pair["base"], pair["donor"], WORK / "out", args)
            tok, m = load_lm(WORK / "out"); R["scores"][mname] = score_all(m, tok)
            print(f"  {mname:<26} " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"][mname].items()) + f"   [{R['secs'][mname]:.0f}s tool time]")
            if mname == f"OMP-{OMP_KS[0]} (official)":      # keep its approximated rows for the 'same direction, calibrated scale' control
                R["info"]["omp_rows_saved"] = True; np.save(OMPF, m.get_input_embeddings().weight.detach().float().cpu().numpy())
            del m; gc.collect(); torch.cuda.empty_cache(); save()
        except Exception as e: print(f"  !!! {mname} failed: {e}"); traceback.print_exc()
    # our ridge, on the same missing rows: first build the same model with ZEROS in the missing rows (official 'zero' method) to find them
    try:
        R["secs"]["zero (official)"] = run_ts(pair["base"], pair["donor"], WORK / "out", ["-a", "zero"])
        tok, m = load_lm(WORK / "out"); V = len(tok)
        E = m.get_input_embeddings().weight.detach().float().cpu().numpy()[:V].copy(); nrm = np.linalg.norm(E, axis=1)
        Z = np.where(nrm < 1e-8)[0]; S = np.where(nrm >= 1e-8)[0]
        if m.get_output_embeddings().weight.data_ptr() != m.get_input_embeddings().weight.data_ptr(): raise RuntimeError("output embeddings are not tied in the tool's output; this notebook handles tied models only")
        print(f"  shared (kept) tokens {len(S)} | missing (approximated) tokens {len(Z)} | vocabulary {V}")
        R["info"].update(n_shared=len(S), n_missing=len(Z), vocab=V)
        if len(Z) == 0: raise RuntimeError("no missing tokens: nothing to approximate")
        dm = AutoModel.from_pretrained(pair["donor"]); D = dm.get_input_embeddings().weight.detach().float().cpu().numpy()[:V]; del dm
        mu_r, mu_d = E[S].mean(0), D[S].mean(0); Rc, Dc = E[S] - mu_r, D[S] - mu_d; rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dz_c = D[Z] - mu_d
        cnt = np.bincount([i for s in TEXT["train_text"] for i in tok.encode(s, add_special_tokens=False) if i < V], minlength=V)
        f_idx = np.argsort(-cnt[S], kind="stable")[:5000]
        res = {}
        t = time.time(); dir_all, raw_all, lam_a = ridge_dirs(Dc, Rc, Dz_c); res["ridge[all]"] = dict(dirs=dir_all, secs=time.time() - t)
        t = time.time(); dir_f, _, lam_f = ridge_dirs(Dc[f_idx], Rc[f_idx], Dz_c); res["ridge[freq5000]"] = dict(dirs=dir_f, secs=time.time() - t)
        print(f"  ridge fit time: all {res['ridge[all]']['secs']:.1f}s, freq5000 {res['ridge[freq5000]']['secs']:.1f}s | lambda {lam_a}, {lam_f}")
        R["secs"]["ridge[all] fit"] = res["ridge[all]"]["secs"]; R["secs"]["ridge[freq5000] fit"] = res["ridge[freq5000]"]["secs"]
        objs = {}
        for rn in ("ridge[all]", "ridge[freq5000]"):
            c, o = calibrate(m, tok, Z, mu_r, rbar, res[rn]["dirs"]); objs[rn] = o; set_rows(m, Z, mu_r + c * rbar * res[rn]["dirs"])
            R["scores"][rn + " (ours, calibrated)"] = score_all(m, tok); R["info"][rn + " scale"] = c
            print(f"  {rn:<18} scale {c} | train objective {o:.4f} | " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R['scores'][rn + ' (ours, calibrated)'].items()))
        sel = min(objs, key=objs.get); R["info"]["sel"] = sel; R["scores"]["ridge[sel] (ours, calibrated)"] = R["scores"][sel + " (ours, calibrated)"]; print("  selected by the training objective:", sel)
        # ablation: ridge with its own predicted length (no calibration)
        set_rows(m, Z, mu_r + raw_all * 1.0); R["scores"]["ridge[all] uncalibrated"] = score_all(m, tok)
        print(f"  ridge[all] uncalibrated: " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"]["ridge[all] uncalibrated"].items()))
        # control: the official OMP direction with our calibrated scale
        if not OMPF.exists():      # official OMP rows were cached in the pkl from an earlier run: run the tool again once to get its rows
            run_ts(pair["base"], pair["donor"], WORK / "out_o", ["-a", "omp", "-k", str(OMP_KS[0])]); _t, _m = load_lm(WORK / "out_o")
            np.save(OMPF, _m.get_input_embeddings().weight.detach().float().cpu().numpy()); del _m; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(WORK / "out_o", ignore_errors=True)
        if OMPF.exists():
            Eo = np.load(OMPF)[:V]; dir_o = unit(Eo[Z] - mu_r)
            c, o = calibrate(m, tok, Z, mu_r, rbar, dir_o); set_rows(m, Z, mu_r + c * rbar * dir_o)
            R["scores"][f"OMP-{OMP_KS[0]} direction, calibrated scale"] = score_all(m, tok); R["info"]["omp_cal_scale"] = c
            print(f"  OMP-{OMP_KS[0]} direction + calibrated scale {c}: " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R['scores'][f'OMP-{OMP_KS[0]} direction, calibrated scale'].items()))
        # ---- fair versions of the official OMP (the tool itself is unchanged) -------------------------------------------------
        # (a) the official OMP-64 rows with the component along the shared-mean direction reset to the shared mean's
        u = mu_r / np.linalg.norm(mu_r); Eo = np.load(OMPF)[:V]; rowsA = Eo[Z] - ((Eo[Z] @ u) - (mu_r @ u))[:, None] * u[None, :]
        set_rows(m, Z, rowsA); nmA = "OMP-64 (official, mean-direction fixed)"; R["scores"][nmA] = score_all(m, tok)
        print(f"  {nmA:<42} " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"][nmA].items()))
        # (b) the official OMP-64 run on mean-centred copies of both models, with the receiver mean added back afterwards
        nmB = "OMP-64 (official, centred inputs)"
        try:
            def shifted_copy(src, dst, sub_in, S_ids=None):
                mdl = AutoModelForCausalLM.from_pretrained(src).float(); tk_ = AutoTokenizer.from_pretrained(src)
                with torch.no_grad():
                    wi = mdl.get_input_embeddings().weight; wi -= torch.as_tensor(sub_in, dtype=wi.dtype)
                    wo = mdl.get_output_embeddings().weight
                    if wo.data_ptr() != wi.data_ptr() and S_ids is not None: wo -= wo[torch.as_tensor(S_ids)].mean(0, keepdim=True)
                if Path(dst).exists(): shutil.rmtree(dst)
                Path(dst).mkdir(parents=True, exist_ok=True); mdl.save_pretrained(dst); tk_.save_pretrained(dst); del mdl; gc.collect()
            shifted_copy(pair["base"], WORK / "c_base", mu_r)                      # receiver: tied, one shift covers input and output
            shifted_copy(pair["donor"], WORK / "c_donor", mu_d, S_ids=S)           # donor: shared-token mean removed from its embeddings
            R["secs"][nmB] = run_ts(WORK / "c_base", WORK / "c_donor", WORK / "out_c", ["-a", "omp", "-k", str(OMP_KS[0])])
            tc, mc = load_lm(WORK / "out_c")
            with torch.no_grad(): w_ = mc.get_input_embeddings().weight; w_[:V] += torch.as_tensor(mu_r, dtype=w_.dtype, device=w_.device)
            R["scores"][nmB] = score_all(mc, tc)
            print(f"  {nmB:<42} " + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"][nmB].items()) + f"   [{R['secs'][nmB]:.0f}s tool time]")
            del mc; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(WORK / "out_c", ignore_errors=True); shutil.rmtree(WORK / "c_base", ignore_errors=True); shutil.rmtree(WORK / "c_donor", ignore_errors=True)
        except Exception as e: print(f"  !!! {nmB} failed: {e}"); traceback.print_exc()
        del m; gc.collect(); torch.cuda.empty_cache(); save()
    except Exception as e: print(f"  !!! ridge part failed: {e}"); traceback.print_exc()
    # official FOCUS (deepfocus): base tokenizer = source, donor tokenizer = target; it builds the whole target embedding table (shared tokens are copied)
    if "FOCUS (official)" not in R["scores"]:
        try:
            from deepfocus import FOCUS
            tok, m = load_lm(WORK / "out"); V = len(tok); btok = AutoTokenizer.from_pretrained(pair["base"])
            src = AutoModelForCausalLM.from_pretrained(pair["base"]).get_input_embeddings().weight.detach().float().cpu()
            t = time.time()
            Fo = FOCUS(target_tokenizer=tok, source_tokenizer=btok, source_embeddings=src, target_training_data_path=str(FOCUS_JSONL), fasttext_model_min_count=FOCUS_MIN_COUNT,
                       device="cpu", verbosity="silent")
            R["secs"]["FOCUS (official)"] = time.time() - t
            if Fo.shape[0] != V: raise RuntimeError(f"FOCUS returned {Fo.shape[0]} rows, expected {V}")
            with torch.no_grad(): m.get_input_embeddings().weight[:V] = Fo.to(m.get_input_embeddings().weight.device, m.get_input_embeddings().weight.dtype)
            R["scores"]["FOCUS (official)"] = score_all(m, tok)
            print("  FOCUS (official)".ljust(28) + ", ".join(f"{k} {v.mean():.2f}" for k, v in R["scores"]["FOCUS (official)"].items()) + f"   [{R['secs']['FOCUS (official)']:.0f}s]")
            del m, src; gc.collect(); torch.cuda.empty_cache(); save()
        except Exception as e: print(f"  !!! FOCUS failed: {e}"); traceback.print_exc()
    shutil.rmtree(WORK / "out", ignore_errors=True); return R

for pair in PAIRS:
    try: run_pair(pair)
    except Exception as e: print(f"!!! pair {pair['name']} failed: {type(e).__name__}: {e}"); traceback.print_exc()
    gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

## Report

In [ ]:
def boot(d, n_boot=2000):
    rng = np.random.default_rng(0); n = len(d); m = d[rng.integers(0, n, (n_boot, n))].mean(1)
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))
REF = f"OMP-{OMP_KS[0]} (official)"
for f in sorted(OUT.glob("*.pkl")):
    R = pickle.load(open(f, "rb")); S = R["scores"]
    print("\n" + "=" * 118 + f"\n{R['pair']} | missing tokens {R['info'].get('n_missing')} of {R['info'].get('vocab')} | selected ridge: {R['info'].get('sel')}\n" + "=" * 118)
    print(f"{'method':<46}{'domain NLL':>12}{'general NLL':>13}   {'domain diff vs ' + REF:<40}{'general diff vs ' + REF}")
    for k, v in S.items():
        line = f"{k:<46}{v['eval_dom'].mean():>12.2f}{(v['eval_gen'].mean() if 'eval_gen' in v else float('nan')):>13.2f}   "
        if REF in S and k not in (REF, "base (own tokenizer)"):
            for key in ("eval_dom", "eval_gen"):
                if key in v and key in S[REF]:
                    m_, lo, hi = boot(v[key] - S[REF][key]); line += f"{m_:+.2f} [{lo:+.2f},{hi:+.2f}]{'*' if lo > 0 or hi < 0 else ' '}".ljust(40)
        print(line)
    print("(negative = better than the official OMP; * = CI excludes 0; 'base (own tokenizer)' is a different tokenization, so it is a reference, not a like-for-like comparison)")
    print("tool time (s): " + ", ".join(f"{k} {v:.0f}" for k, v in R["secs"].items()))